# Food Production Challenge 1 - Baseline Submission

This notebook provides a simple baseline for **Food Production Challenge 1: Shelf Life Prediction**.

**Goal**: Predict `shelf_life_remaining_days` for each production batch
**Metric**: Mean Absolute Error (MAE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular batch data with a simple Random Forest regressor.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your-api-key-here",        # Get from your team dashboard
    team_name="your-team-name-here"     # Your exact team name
)

# Load data from PVC paths
print("📂 Loading Food Production Challenge 1 data...")

# Load batch data
train_batches = pd.read_csv("/home/jovyan/shared/datasets/FoodProduction/batches_train.csv")
test_batches = pd.read_csv("/home/jovyan/shared/datasets/FoodProduction/batches_test.csv")

print(f"✅ Data loaded:")
print(f"   Train batches: {train_batches.shape}")
print(f"   Test batches: {test_batches.shape}")
print(f"   Train columns: {list(train_batches.columns)}")
print(f"   Test columns: {list(test_batches.columns)}")


In [ ]:
# 2. Tabular-Only Baseline Model and Predictions

# From data inspection - batches columns:
# batch_id, sku_id, site_id, dwell_hours, mean_temp_F, mean_rh_pct, door_opens_count, shelf_life_remaining_days (train only)

# Select numeric production features for baseline
batch_features = ['sku_id', 'site_id', 'dwell_hours', 'mean_temp_F', 'mean_rh_pct', 'door_opens_count']
print(f"📊 Using batch features: {batch_features}")

# Prepare training data
X_train = train_batches[batch_features].fillna(0)
y_train = train_batches['shelf_life_remaining_days']  # Target variable

# Prepare test data
X_test = test_batches[batch_features].fillna(0)

# Train simple Random Forest baseline
print("🤖 Training Random Forest regressor...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
predictions = model.predict(X_test)

# Create submission file (format: batch_id,shelf_life_remaining_days)
submission_df = pd.DataFrame({
    'batch_id': test_batches['batch_id'],
    'shelf_life_remaining_days': predictions
})

# Save predictions
submission_df.to_csv("foodproduction_challenge1_predictions.csv", index=False)
print(f"✅ Predictions saved: {submission_df.shape[0]} predictions")
print(f"   Preview: {submission_df.head(3)}")
print(f"   Shelf life range: {predictions.min():.1f} to {predictions.max():.1f} days")


In [ ]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("FoodProduction", 1, "foodproduction_challenge1_predictions.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Food Production Challenge 2!")
